<a href="https://colab.research.google.com/github/Kiruthika2006-devil/AI_Based_Disease_Prediction_System/blob/main/Disease_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**1.IMPORT THE NECESSARY LIBRARIES **

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, SimpleRNN, Embedding

**2. LOAD DATA**

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/Kiruthika2006-devil/AI_Based_Disease_Prediction_System/main/disease_dataset.csv")


**3.PREPROCESS THE DATA**

In [ ]:
print(df.head())

        Fever  Headache     Cough   Fatigue  Body_Pain      Disease
0   98.283156  4.118632  2.687983  4.116661   8.470956    Body Ache
1   98.284660  6.963971  1.960734  5.495257   8.033919    Body Ache
2  103.622552  8.702915  2.854463  6.546993   2.205383      Malaria
3   98.254732  1.354851  5.880676  5.418289   2.695560  Common Cold
4   97.182670  0.357962  6.206723  4.488365   1.406323        Cough


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Fever      5000 non-null   float64
 1   Headache   5000 non-null   float64
 2   Cough      5000 non-null   float64
 3   Fatigue    5000 non-null   float64
 4   Body_Pain  5000 non-null   float64
 5   Disease    5000 non-null   object 
dtypes: float64(5), object(1)
memory usage: 234.5+ KB


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:

# Drop completely empty columns if they exist
df = df.dropna(axis=1, how='all')

In [ ]:
# Assume the last column contains the disease names (e.g., 'prognosis' or 'disease')
X = df.iloc[:, :-1].values  # Features (Symptoms: 0s and 1s)
y = df.iloc[:, -1].values   # Target (Disease Labels)
print(X,y)

[[ 98.28315625   4.11863151   2.68798308   4.11666147   8.47095604]
 [ 98.28466015   6.96397103   1.96073356   5.49525654   8.03391883]
 [103.6225524    8.70291458   2.85446277   6.54699346   2.20538253]
 ...
 [103.11256915   7.83445141   1.00620029   9.0943798    5.43887927]
 [ 97.43205321   1.613986     1.82666601   2.08997452   1.24789082]
 [ 97.66109426   1.99477253   5.15527253   4.70163002   1.79959698]] ['Body Ache' 'Body Ache' 'Malaria' ... 'Dengue' 'Runny Nose' 'Common Cold']


In [ ]:


try:
    df = df
    print("Dataset loaded successfully!")
except Exception as e:
    print(f"Could not load from URL, please ensure the path is correct. Error: {e}")
    # Fallback to local file if needed: df = pd.read_csv('disease_dataset.csv')

# Encode categorical text labels into integers
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
num_classes = len(np.unique(y_encoded))

# Split into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

print(f"Number of symptoms (features): {X.shape[1]}")
print(f"Number of target diseases (classes): {num_classes}")


# 2. MODEL 2: RECURRENT NEURAL NETWORK (RNN)
print("\n--- Training Recurrent Neural Network (RNN) ---")

# For RNN, data must be 3D: (samples, timesteps, features)
# We treat each symptom feature as a timestep with 1 element
X_train_rnn = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
X_test_rnn = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))

rnn_model = Sequential([
    SimpleRNN(64, activation='tanh', input_shape=(X_train_rnn.shape[1], 1), return_sequences=False),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])

rnn_model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

# Train the RNN
rnn_model.fit(X_train_rnn, y_train, epochs=15, batch_size=64, validation_split=0.1, verbose=1)

# Evaluate RNN
rnn_preds = np.argmax(rnn_model.predict(X_test_rnn), axis=-1)
rnn_acc = accuracy_score(y_test, rnn_preds)
print(f"\nRNN Test Accuracy: {rnn_acc * 100:.2f}%")

# 3. FINAL COMPARISON REPORT
print("\n=== Final Performance Comparison ===")
print(f"RNN Accuracy: {rnn_acc * 100:.2f}%")

print("\nDetailed RNN Classification Report:")
print(classification_report(y_test, rnn_preds, target_names=label_encoder.classes_))

Dataset loaded successfully!
Number of symptoms (features): 5
Number of target diseases (classes): 8

--- Training Recurrent Neural Network (RNN) ---


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - accuracy: 0.5008 - loss: 1.3757 - val_accuracy: 0.7400 - val_loss: 0.7323
Epoch 2/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7375 - loss: 0.6534 - val_accuracy: 0.7925 - val_loss: 0.4860
Epoch 3/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7906 - loss: 0.4998 - val_accuracy: 0.8250 - val_loss: 0.4123
Epoch 4/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8042 - loss: 0.4451 - val_accuracy: 0.8300 - val_loss: 0.3924
Epoch 5/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8158 - loss: 0.4277 - val_accuracy: 0.8425 - val_loss: 0.3704
Epoch 6/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8253 - loss: 0.4122 - val_accuracy: 0.8500 - val_loss: 0.3903
Epoch 7/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8308 - loss: 0.3916 - val_accuracy: 0.8350 - val_loss: 0.3558
Epoch 8/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8283 - loss: 0.3941 - val_accuracy: 0.8475 - val_loss